# Course 4: Large Language Models
## Session 1: The modern transformer

**Goal:** build the pieces of a GPT (attention, the causal mask, rotary positions), then train a tiny one and sample from it.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

### 1. Attention in eight lines
Each position asks a question (query), every position offers a key and a value, and the answer is a weighted average of the values. The causal mask stops a position looking at later ones.

In [ ]:
import math, torch, torch.nn.functional as F
torch.manual_seed(0)

def attention(q, k, v, causal=True):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    if causal:
        T = q.size(-2)
        scores = scores.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), float("-inf"))
    w = torch.softmax(scores, dim=-1)
    return w @ v, w

q, k, v = torch.randn(3, 1, 5, 8).unbind(0)
out, w = attention(q, k, v)
print("attention weights of position 2:", w[0, 2].numpy().round(3), "(zeros after position 2)")
assert torch.allclose(out, F.scaled_dot_product_attention(q, k, v, is_causal=True), atol=1e-6)
print("matches PyTorch's built-in attention")

### 2. The causal mask means the past cannot see the future
Change the last token and check that the outputs at earlier positions do not move.

In [ ]:
x = torch.randn(1, 5, 8); x2 = x.clone(); x2[0, 4] += 1.0
a, _ = attention(x, x, x); b, _ = attention(x2, x2, x2)
print("change at earlier positions:", (a[0, :4] - b[0, :4]).abs().max().item(), "| at the last position:", (a[0, 4] - b[0, 4]).abs().max().item())
assert torch.allclose(a[0, :4], b[0, :4])

### 3. Rotary position embeddings (RoPE)
Rotate each pair of dimensions of the query and key by an angle that grows with position. The dot product then depends only on how far apart two positions are.

In [ ]:
def rope(x, pos):
    d = x.size(-1); freqs = 1.0 / (10000 ** (torch.arange(0, d, 2) / d))
    ang = pos[:, None] * freqs[None, :]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * ang.cos() - x2 * ang.sin(), x1 * ang.sin() + x2 * ang.cos()], -1).flatten(-2)

qv, kv = torch.randn(8), torch.randn(8)
score = lambda m, n: (rope(qv[None], torch.tensor([float(m)])) @ rope(kv[None], torch.tensor([float(n)])).T).item()
print("positions (3, 1): %.4f | positions (10, 8): %.4f | positions (10, 5): %.4f" % (score(3, 1), score(10, 8), score(10, 5)))
assert abs(score(3, 1) - score(10, 8)) < 1e-4            # same distance, same score

### 4. A full GPT
Pre-norm blocks, attention plus a feed-forward layer, residual connections, and tied input and output embeddings. This model uses learned positions; swapping in RoPE is a TODO.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

### 5. Train it on a toy Bangla language
The corpus is generated from a small grammar, so we can check exactly whether a generated sentence is valid, and we know the best possible loss.

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

In [ ]:
print("example:", " ".join(train_sents[0]), "| vocabulary", len(VOCAB), "| training sentences", len(train_sents))
cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=64)
model = train_lm(cfg, steps=600)
floor = math.log(len(ALL_SENTENCES)) / 6           # 1,600 equally likely sentences of 6 tokens each
print("parameters: %d | validation loss %.3f | best possible loss %.3f" % (sum(p.numel() for p in model.parameters()), val_loss(model), floor))
samples = sample_sentences(model, 200)
for s in samples[:4]: print("  ", " ".join(s))
valid = sum(map(is_valid, samples)) / 200
new = sum(s in val_sents for s in samples) / 200
print(f"valid sentences: {valid:.2f} | sentences never seen in training: {new:.2f}")
assert valid > 0.9 and val_loss(model) < floor + 0.15

The model writes valid sentences that were not in its training data. It has learned the rule, not the list.

### TODO
1. Replace the learned position embedding with RoPE inside `Block.attn` (rotate `q` and `k` after splitting into heads).
2. Swap LayerNorm for RMSNorm and GELU for SwiGLU. Does the loss change at this scale?
3. Mixture of experts: replace the feed-forward layer with 4 small experts and a router that picks the top 1. Count the parameters that are active per token.